In [2]:
!pip install kneed

In [12]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from datetime import timedelta
from sklearn.preprocessing import StandardScaler
from kneed import KneeLocator
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples
from mpl_toolkits.mplot3d import Axes3D

In [15]:
df=pd.read_csv("D:\Downloads\Details.csv")
print(f"\nData size: {df.shape}")
df.head()


Data size: (1500, 7)


,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode
0,B-25681,1096,658,7,Electronics,Electronic Games,COD
1,B-26055,5729,64,14,Furniture,Chairs,EMI
2,B-25955,2927,146,8,Furniture,Bookcases,EMI
3,B-26093,2847,712,8,Electronics,Printers,Credit Card
4,B-25602,2617,1151,4,Electronics,Phones,Credit Card


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1500 non-null   object
 1   Amount        1500 non-null   int64 
 2   Profit        1500 non-null   int64 
 3   Quantity      1500 non-null   int64 
 4   Category      1500 non-null   object
 5   Sub-Category  1500 non-null   object
 6   PaymentMode   1500 non-null   object
dtypes: int64(3), object(4)
memory usage: 82.2+ KB


In [17]:
df.describe()

,Amount,Profit,Quantity
count,1500.000000,1500.00000,1500.000000
mean,291.847333,24.64200,3.743333
std,461.924620,168.55881,2.184942
min,4.000000,-1981.00000,1.000000
25%,47.750000,-12.00000,2.000000
50%,122.000000,8.00000,3.000000
75%,326.250000,38.00000,5.000000
max,5729.000000,1864.00000,14.000000


In [21]:
df_order=pd.read_csv(r"D:\Downloads\archive\Orders.csv")
print(f"\nData size: {df_order.shape}")
df_order.head()


Data size: (500, 5)


,Order ID,Order Date,CustomerName,State,City
0,B-26055,10-03-2018,Harivansh,Uttar Pradesh,Mathura
1,B-25993,03-02-2018,Madhav,Delhi,Delhi
2,B-25973,24-01-2018,Madan Mohan,Uttar Pradesh,Mathura
3,B-25923,27-12-2018,Gopal,Maharashtra,Mumbai
4,B-25757,21-08-2018,Vishakha,Madhya Pradesh,Indore


In [22]:
df_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      500 non-null    object
 1   Order Date    500 non-null    object
 2   CustomerName  500 non-null    object
 3   State         500 non-null    object
 4   City          500 non-null    object
dtypes: object(5)
memory usage: 19.7+ KB


In [24]:
df_order.describe()

,Order ID,Order Date,CustomerName,State,City
count,500,500,500,500,500
unique,500,307,336,19,25
top,B-26055,24-11-2018,Shreya,Maharashtra,Indore
freq,1,7,6,94,71


## preprocessing

In [27]:
df_sales=pd.merge(df,df_order,on="Order ID", how='left')
df_sales

,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order Date,CustomerName,State,City
0,B-25681,1096,658,7,Electronics,Electronic Games,COD,04-06-2018,Bhawna,Madhya Pradesh,Indore
1,B-26055,5729,64,14,Furniture,Chairs,EMI,10-03-2018,Harivansh,Uttar Pradesh,Mathura
2,B-25955,2927,146,8,Furniture,Bookcases,EMI,16-01-2018,Shiva,Maharashtra,Pune
3,B-26093,2847,712,8,Electronics,Printers,Credit Card,27-03-2018,Sarita,Maharashtra,Pune
4,B-25602,2617,1151,4,Electronics,Phones,Credit Card,01-04-2018,Vrinda,Maharashtra,Pune
...,...,...,...,...,...,...,...,...,...,...,...
1495,B-25700,7,-3,2,Clothing,Hankerchief,COD,25-06-2018,Shubhi,Maharashtra,Mumbai
1496,B-25757,3151,-35,7,Clothing,Trousers,EMI,21-08-2018,Vishakha,Madhya Pradesh,Indore
1497,B-25973,4141,1698,13,Electronics,Printers,COD,24-01-2018,Madan Mohan,Uttar Pradesh,Mathura
1498,B-25698,7,-2,1,Clothing,Hankerchief,COD,23-06-2018,Amisha,Tamil Nadu,Chennai


In [28]:
df_sales.isnull().sum()

Order ID        0
Amount          0
Profit          0
Quantity        0
Category        0
Sub-Category    0
PaymentMode     0
Order Date      0
CustomerName    0
State           0
City            0
dtype: int64

In [30]:
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1500 non-null   object
 1   Amount        1500 non-null   int64 
 2   Profit        1500 non-null   int64 
 3   Quantity      1500 non-null   int64 
 4   Category      1500 non-null   object
 5   Sub-Category  1500 non-null   object
 6   PaymentMode   1500 non-null   object
 7   Order Date    1500 non-null   object
 8   CustomerName  1500 non-null   object
 9   State         1500 non-null   object
 10  City          1500 non-null   object
dtypes: int64(3), object(8)
memory usage: 129.0+ KB


In [31]:
df_sales.columns=df_sales.columns.str.strip()

In [32]:
print(df_sales.columns.tolist())

['Order ID', 'Amount', 'Profit', 'Quantity', 'Category', 'Sub-Category', 'PaymentMode', 'Order Date', 'CustomerName', 'State', 'City']


In [34]:
df_sales.describe(exclude=np.number)

,Order ID,Category,Sub-Category,PaymentMode,Order Date,CustomerName,State,City
count,1500,1500,1500,1500,1500,1500,1500,1500
unique,500,3,17,5,307,336,19,25
top,B-26056,Clothing,Saree,COD,10-03-2018,Abhishek,Madhya Pradesh,Indore
freq,12,949,211,684,33,21,314,248


In [37]:
# copying the Dataframe
df=df_sales.copy()
df.columns=df.columns.str.strip()

In [41]:
# conversion of invoicedate
df["Order Date"]=pd.to_datetime(df["Order Date"], dayfirst=True)
reference_date=df["Order Date"].max() + pd.Timedelta(days=1)

## RFM Analysis

In [44]:
# calculate rfm
df_rfm=df.groupby("CustomerName").agg({"Order Date":"max",
                                       "Order ID": "nunique",
                                       "Amount":"sum"
                                       }).reset_index()
df_rfm.rename(columns={"Order Date":"LastPurchase",
                       "Order ID":"Frequency",
                       "Amount":"Monetary"
                       },inplace=True)
df_rfm["recency"]=(reference_date-df_rfm["LastPurchase"]).dt.days
df_rfm.head()

,CustomerName,LastPurchase,Frequency,Monetary,recency
0,Aakanksha,2018-07-01,1,74,184
1,Aarushi,2018-04-08,3,4701,268
2,Aastha,2018-10-26,1,3276,67
3,Aayush,2018-11-15,1,556,47
4,Aayushi,2018-09-15,3,689,108


In [48]:
# calculate rfm
#Recency
df_rfm["R_score"]=pd.qcut(df_rfm["recency"].rank(method="first"),
                          4,
                          labels=[4,3,2,1]
                          ).astype(int)
# Frequency
df_rfm["F_Score"]=pd.qcut(df_rfm["Frequency"].rank(method="first"),
                          4,
                          labels=[1,2,3,4]
                          ).astype(int)
# monetary
df_rfm["M_Score"]=pd.qcut(df_rfm["Monetary"].rank(method="first"),
                          4,
                          labels=[1,2,3,4]
                          ).astype(int)


In [50]:
# rfm total score
df_rfm["rfm_score"]=(df_rfm["R_score"]*100+
                     df_rfm["F_Score"]*10+
                     df_rfm["M_Score"]
                     )